# Building a knowledge base from PDF documentation using Databricks Vector search (RAG)

As we saw before, our agent isn't working well when it comes to answer specific, technical questions such as WIFI router error code.

That's because it doesn't have any knowledge about our internal systems and product. 

Thanksfully, all this information is available to us as PDF. These pdf are stored in our volume. 

We'll parse them and save them in our Vector Search, and then add a retriever to our agent to improve its capabilities!


<div style="background-color: #d4e7ff; padding: 10px; border-radius: 15px;">
<strong>Note:</strong> Coming soon, we'll show how to add a Knowledge base in a few clicks leveraging Databricks Agents!
</div>

<!-- Collect usage data (view). Remove it to disable collection or disable tracker during installation. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=data-science&org_id=2162748966026566&notebook=%2F03-knowledge-base-rag%2F03.1-pdf-rag-tool&demo_name=ai-agent&event=VIEW&path=%2F_dbdemos%2Fdata-science%2Fai-agent%2F03-knowledge-base-rag%2F03.1-pdf-rag-tool&version=1">


In [0]:
%pip install -U -qqqq mlflow>=3.1.4 langchain==0.3.27 langgraph==0.6.11 databricks-langchain pydantic databricks-agents unitycatalog-langchain[databricks] databricks-feature-engineering==0.12.1 protobuf<5  cryptography<43 databricks-mcp
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%run ../_resources/01-setup

USE CATALOG `main_build`
using catalog.database `main_build`.`dbdemos_ai_agent`


data already exists



## 1. Extracting the PDF information
Databricks provides a builtin `ai_parse_document` function, leveraging AI to analyze and extract PDF information as text. This makes it super easy to ingest unstructured information!

In [0]:
%sql
SELECT path FROM READ_FILES('/Volumes/main/dbdemos_ai_agent/raw_data/pdf_documentation/', format => 'binaryFile') limit 2

path
dbfs:/Volumes/main/dbdemos_ai_agent/raw_data/pdf_documentation/sat_survey_2024_manual.pdf
dbfs:/Volumes/main/dbdemos_ai_agent/raw_data/pdf_documentation/accessibility_2024_manual.pdf


In [0]:
%sql
-- ai_parse_document is available in DBR 17.1 or serverless runtime
SELECT ai_parse_document(content) AS parsed_document
  FROM READ_FILES('/Volumes/main/dbdemos_ai_agent/raw_data/pdf_documentation/', format => 'binaryFile') limit 2

parsed_document {"document":{"elements":[{"bbox":[{"coord":[142,147,850,183],"page_id":0}],"content":"Customer Satisfaction Survey Procedures","description":null,"id":0,"type":"title"},{"bbox":[{"coord":[394,200,592,219],"page_id":0}],"content":"Model: SAT-SURVEY-2024","description":null,"id":1,"type":"text"},{"bbox":[{"coord":[386,232,603,252],"page_id":0}],"content":"Category: Customer Service","description":null,"id":2,"type":"text"},{"bbox":[{"coord":[343,261,645,280],"page_id":0}],"content":"Version: 1.0 (Effective Date: April 2024)","description":null,"id":3,"type":"text"},{"bbox":[{"coord":[90,362,343,390],"page_id":0}],"content":"Table of Contents","description":null,"id":4,"type":"section_header"},{"bbox":[{"coord":[93,415,267,435],"page_id":0}],"content":"1. Executive Summary","description":null,"id":5,"type":"section_header"},{"bbox":[{"coord":[93,442,293,460],"page_id":0}],"content":"2. Technical Specifications","description":null,"id":6,"type":"section_header"},{"bbox":[{"coord":[93,467,357,487],"page_id":0}],"content":"3. Installation & Setup Instructions","description":null,"id":7,"type":"section_header"},{"bbox":[{"coord":[93,493,385,512],"page_id":0}],"content":"4. Configuration & Management Guide","description":null,"id":8,"type":"section_header"},{"bbox":[{"coord":[93,518,276,539],"page_id":0}],"content":"5. Error Code Reference","description":null,"id":9,"type":"section_header"},{"bbox":[{"coord":[93,545,237,564],"page_id":0}],"content":"6. Troubleshooting","description":null,"id":10,"type":"section_header"},{"bbox":[{"coord":[93,572,448,590],"page_id":0}],"content":"7. Maintenance & Firmware Update Procedures","description":null,"id":11,"type":"section_header"},{"bbox":[{"coord":[93,597,259,617],"page_id":0}],"content":"8. Network Diagrams","description":null,"id":12,"type":"section_header"},{"bbox":[{"coord":[93,623,345,644],"page_id":0}],"content":"9. Performance Optimization Tips","description":null,"id":13,"type":"section_header"},{"bbox":[{"coord":[93,650,447,669],"page_id":0}],"content":"10. Compliance, Regulatory & Safety Warnings","description":null,"id":14,"type":"section_header"},{"bbox":[{"coord":[93,676,293,695],"page_id":0}],"content":"11. Security Configuration","description":null,"id":15,"type":"section_header"},{"bbox":[{"coord":[93,702,381,722],"page_id":0}],"content":"12. Compatibility & Integration Matrix","description":null,"id":16,"type":"section_header"},{"bbox":[{"coord":[93,728,407,747],"page_id":0}],"content":"13. Warranty, Return, and Refund Policies","description":null,"id":17,"type":"section_header"},{"bbox":[{"coord":[93,755,332,774],"page_id":0}],"content":"14. Frequently Asked Questions","description":null,"id":18,"type":"section_header"},{"bbox":[{"coord":[93,780,331,800],"page_id":0}],"content":"15. Glossary of Technical Terms","description":null,"id":19,"type":"section_header"},{"bbox":[{"coord":[93,806,353,825],"page_id":0}],"content":"16. Support & Escalation Contacts","description":null,"id":20,"type":"section_header"},{"bbox":[{"coord":[93,831,245,852],"page_id":0}],"content":"17. Revision History","description":null,"id":21,"type":"section_header"},{"bbox":[{"coord":[93,930,412,961],"page_id":0}],"content":"1. Executive Summary","description":null,"id":22,"type":"section_header"},{"bbox":[{"coord":[93,982,887,1107],"page_id":0}],"content":"The \"Customer Satisfaction Survey Procedures\" document provides comprehensive guidelines for administering, managing, and following up on customer satisfaction surveys related to the SAT-SURVEY-2024 model. This manual ensures standardized processes to accurately gauge customer feedback, analyze data, and implement improvements. It covers all aspects from survey deployment, data collection, analysis, troubleshooting, and compliance, ensuring consistency and quality in customer service initiatives.","description":null,"id":23,"type":"text"},{"bbox":[{"coord":[93,1118,891,1181],"page_id":0}],"content":"The procedures outlined herein 

## 1.1/ Create our knowledge base table

Let's first create our table. We'll enable Change Data Feed so that we can create our vector search on top of it.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS knowledge_base (
  id BIGINT GENERATED ALWAYS AS IDENTITY,
  product_name STRING,
  title STRING,
  content STRING,
  doc_uri STRING)
  TBLPROPERTIES (delta.enableChangeDataFeed = true);


## 1.2/ PDF to text with ai_parse_document

Let's now use Databricks built in `ai_parse_document` function to automatically parse the PDF document for us, making it super easy to extract the information!

*Note: in this case, we have relatively small pdf documents, so we'll merge all the pages of the document in one single text field for our RAG system to work properly. Bigger docs might need some pre-processing steps to potentially reduce context size and be able to search/retreive more documents, adding potential pre-processing steps, for example ensuring the WIFI Router model is present in all the chunk to keep the vector search more relevant.*

In [0]:
%sql
INSERT OVERWRITE TABLE knowledge_base (product_name, title, content, doc_uri)
SELECT ai_extract.product_name, ai_extract.title, content, doc_uri
FROM (
  SELECT
    ai_extract(content, array('product_name', 'title')) AS ai_extract,
    content,
    doc_uri
  FROM (
    SELECT array_join(
            transform(parsed_document:document.elements::ARRAY<STRUCT<content:STRING>>, x -> x.content), '\n') AS content,
           path as doc_uri
    FROM (
      SELECT ai_parse_document(content) AS parsed_document, path
      FROM READ_FILES('/Volumes/main/dbdemos_ai_agent/raw_data/pdf_documentation/', format => 'binaryFile') 
      LIMIT 5 -- ADDED FIX LIMIT FOR DEMO COST - DROP IT IN REAL WORKLOAD
    )
  )
);

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
SELECT * FROM knowledge_base;

id product_name title content doc_uri 56 SAT-SURVEY-2024 Customer Satisfaction Survey Procedures Customer Satisfaction Survey Procedures
Model: SAT-SURVEY-2024
Category: Customer Service
Version: 1.0 (Effective Date: April 2024)
Table of Contents
1. Executive Summary
2. Technical Specifications
3. Installation & Setup Instructions
4. Configuration & Management Guide
5. Error Code Reference
6. Troubleshooting
7. Maintenance & Firmware Update Procedures
8. Network Diagrams
9. Performance Optimization Tips
10. Compliance, Regulatory & Safety Warnings
11. Security Configuration
12. Compatibility & Integration Matrix
13. Warranty, Return, and Refund Policies
14. Frequently Asked Questions
15. Glossary of Technical Terms
16. Support & Escalation Contacts
17. Revision History
1. Executive Summary
The "Customer Satisfaction Survey Procedures" document provides comprehensive guidelines for administering, managing, and following up on customer satisfaction surveys related to the SAT-SURVEY-2024 model. This manual ensures standardized processes to accurately gauge customer feedback, analyze data, and implement improvements. It covers all aspects from survey deployment, data collection, analysis, troubleshooting, and compliance, ensuring consistency and quality in customer service initiatives.
The procedures outlined herein are designed for use by customer service representatives, technical staff, and management teams to facilitate effective survey operations, ensure data integrity, and uphold regulatory standards.
Page
2. Technical Specifications
 Parameter Specification Model Number SAT-SURVEY-2024 Device Type Customer Satisfaction Survey Management Module Supported Platforms Web-based interface, Mobile app (iOS & Android) Survey Capacity Up to 10,000 simultaneous respondents Data Storage Secure cloud storage with 99.9% uptime SLA Security SSL/TLS encryption, Role-based access control, GDPR compliant Connectivity Ethernet, Wi-Fi (802.11ac/n), Cellular (LTE/5G) Supported Languages English, Spanish, French, German, Chinese Response Time Survey deployment within 5 seconds; Data retrieval within 2 seconds Compliance ISO 27001, GDPR, HIPAA (optional modules) 
3. Installation & Setup Instructions
3.1 Environment Requirements
Server Environment: Linux-based OS (Ubuntu 20.04 LTS or higher) or Windows Server 2019+
Web Browser Compatibility: Latest versions of Chrome, Firefox, Edge, Safari
Network: Stable internet connection with minimum bandwidth of 10 Mbps
Security: SSL certificate installed for HTTPS access
3.2 Hardware Requirements
Processor: Quad-core 2.5 GHz or higher
Memory: Minimum 8 GB RAM
Storage: At least 100 GB SSD for local deployment; cloud recommended
Network Interface: Ethernet port or Wi-Fi adapter
3.3 Software Installation Steps
1. Download the latest installation package from the official portal.
2. Ensure all prerequisites are met (see environment requirements).
3. Run the installer with administrator privileges.
4. Follow the on-screen prompts to complete installation.
5. Configure network settings: assign static IP, set DNS as per network policy.
6. Secure the system by installing SSL certificates and configuring firewalls.
7. Access the web interface via https://port.
3.4 Initial Configuration
1. Login with default admin credentials provided in the setup guide.
2. Change default password immediately after first login.
3 Configure survey parameters: language options, survey templates, respondent limits
4. Set up user roles and permissions under Settings > User Management.
5. Integrate with existing CRM or customer databases if applicable
6. Test survey deployment with a sample respondent account.
4. Configuration & Management Guide
4.1 User Management
Navigate to Settings > User Management to add, modify, or remove user accounts. Assign roles such as Administrator, Supervisor, or Respondent.
4.2 Survey Design & Deployment
1. Access Survey Builder from the main menu.
2. Create new survey templates or modify existing ones.
3. 


## 2/ Create our vector search table

### 2.1/ Vector search Endpoints

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/chatbot-rag/rag-basic-prep-2.png?raw=true" style="float: right; margin-left: 10px" width="400px">

Vector search endpoints are entities where your indexes will live. Think about them as entry point to handle your search request. 

Let's start by creating our first Vector Search endpoint. Once created, you can view it in the [Vector Search Endpoints UI](#/setting/clusters/vector-search). Click on the endpoint name to see all indexes that are served by the endpoint.

In [0]:
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient(disable_notice=True)

if not endpoint_exists(vsc, VECTOR_SEARCH_ENDPOINT_NAME):
    vsc.create_endpoint(name=VECTOR_SEARCH_ENDPOINT_NAME, endpoint_type="STANDARD")

wait_for_vs_endpoint_to_be_ready(vsc, VECTOR_SEARCH_ENDPOINT_NAME)
print(f"Endpoint named {VECTOR_SEARCH_ENDPOINT_NAME} is ready.")

Endpoint named dbdemos_vs_endpoint is ready.



<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/chatbot-rag/rag-basic-prep-3.png?raw=true" style="float: right; margin-left: 10px" width="400px">


### 2.2/ Creating the Vector Search Index

Once the endpoint is created, all we now have to do is to as Databricks to create the index on top of the existing table. 

You just need to specify the text column and our embedding foundation model (`GTE`).  Databricks will build and synchronize the index automatically for us.

Note that Databricks provides 3 type of vector search:

* **Managed embeddings**: Databricks creates the embeddings for you from a text field and Databricks synchronize the Delta table to your index (what we'll use)
* **Self managed embeddings**: You compute the embeddings yourself and save them to your Delta table  and Databricks synchronize the Delta table to your index
* **Direct access**: you manage the VS indexation yourself (no Delta table)

This can be done using the API, or in a few clicks within the Unity Catalog Explorer menu:

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/index_creation.gif?raw=true" width="600px">


In [0]:
from databricks.sdk import WorkspaceClient

#The table we'd like to index
source_table_fullname = f"{catalog}.{dbName}.knowledge_base"
# Where we want to store our index
vs_index_fullname = f"{catalog}.{dbName}.knowledge_base_vs_index"

if not index_exists(vsc, VECTOR_SEARCH_ENDPOINT_NAME, vs_index_fullname):
  print(f"Creating index {vs_index_fullname} on endpoint {VECTOR_SEARCH_ENDPOINT_NAME}...")
  vsc.create_delta_sync_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    index_name=vs_index_fullname,
    source_table_name=source_table_fullname,
    pipeline_type="TRIGGERED",
    primary_key="id",
    embedding_source_column='content', #The column containing our text
    embedding_model_endpoint_name='databricks-gte-large-en' #The embedding endpoint used to create the embeddings
  )
  #Let's wait for the index to be ready and all our embeddings to be created and indexed
  wait_for_index_to_be_ready(vsc, VECTOR_SEARCH_ENDPOINT_NAME, vs_index_fullname)
else:
  #Trigger a sync to update our vs content with the new data saved in the table
  wait_for_index_to_be_ready(vsc, VECTOR_SEARCH_ENDPOINT_NAME, vs_index_fullname)
  vsc.get_index(VECTOR_SEARCH_ENDPOINT_NAME, vs_index_fullname).sync()

print(f"index {vs_index_fullname} on table {source_table_fullname} is ready")

index main.dbdemos_ai_agent.knowledge_base_vs_index on table main.dbdemos_ai_agent.knowledge_base is ready


## 2.3/ Try our VS index: searching for relevant content

That's all we have to do. Databricks will automatically capture and synchronize new entries in your table with the index.

Note that depending on your dataset size and model size, index creation can take a few seconds to start and index your embeddings.

Let's give it a try and search for similar content.

*Note: `similarity_search` also support a filters parameter. This is useful to add a security layer to your RAG system: you can filter out some sensitive content based on who is doing the call (for example filter on a specific department based on the user preference).*

In [0]:
question = "My wifi router gives me error 01, what should I do?"

results = vsc.get_index(VECTOR_SEARCH_ENDPOINT_NAME, vs_index_fullname).similarity_search(
  query_text=question,
  columns=["id", "content"],
  num_results=1)
docs = results.get('result', {}).get('data_array', [])
docs

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


[[54.0,
  "Table of Contents\n1. Executive Summary Smart Home Network Setup\n2. Technical Specifications Model: SMART-HOME-NET | Version: 1.0\n3. Installation & Setup Instructions\n4. Configuration & Management Guide\n5. Error Code Reference\n6. Troubleshooting\n7. Maintenance & Firmware Updates\n8. Network Diagrams\n9. Performance Optimization Tips\n10. Compliance, Safety & Regulatory\n11. Security Configuration\n12. Compatibility & Integration Matrix\n13. Warranty, Return & Refund Policies\n14. Frequently Asked Questions\n15. Glossary of Terms\n16. Support & Escalation Contacts\n17. Revision History\nPage of\n1. Executive Summary Home Network Setup\nThe Smart Home Network Setup for the SMART-HOME-NET device provides a comprehensive solution for integrating and managing smart home devices within a secure, reliable, and high-performance network environment. This manual details the installation, configuration, troubleshooting, and maintenance procedures necessary for optimal operation. 

## 3/ Update our existing Agent to add the retriever as new tool

Now that our index is ready, all we have to do is to add it as retriever to our existing agent!

We'll reuse the `agent.py` and `agent_config.yaml` file: simply add the retriever configuration and our agent will add it as one of the tools available!

In [0]:
import mlflow
import yaml, sys, os
import mlflow.models
# Add the ../agent_eval path relative to current working directory
agent_eval_path = os.path.abspath(os.path.join(os.getcwd(), "../02-agent-eval"))
sys.path.append(agent_eval_path)
# Let's also use the same experiment as in our previous notebook to keep all the trace in a single place
mlflow.set_experiment(agent_eval_path+"/02.1_agent_evaluation")
conf_path = os.path.join(agent_eval_path, 'agent_config.yaml')

try:
    config = yaml.safe_load(open(conf_path))
    config["config_version_name"] = "model_with_retriever"
    config["retriever_config"] =  {
        "index_name": vs_index_fullname,
        "tool_name": "product_technical_docs_retriever",
        "num_results": 1,
        "description": "Retrieves internal documentation about our products, infrastructure, router and other, including features, usage, and troubleshooting. Use this tool for any questions about product documentation or product issues."
    }
    yaml.dump(config, open(conf_path, "w"))
except Exception as e:
    print(f"Skipped update - ignore for job run - {e}")

model_config = mlflow.models.ModelConfig(development_config=conf_path)

Skipped update - ignore for job run - [Errno 22] Invalid argument


In [0]:
%pip install databricks-mcp

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from agent import AGENT 

#Let's try our retriever to make sure we know have access to the wifi router pdf guide
request_example = "How do I restart my WIFI router ADSL-R500?"
answer = AGENT.predict({"input":[{"role": "user", "content": request_example}]})

2025/11/12 22:08:58 WARNING mlflow.pyfunc: You have manually traced predict with @mlflow.trace, but this is unnecessary with ResponsesAgent subclasses. You can remove the @mlflow.trace decorator and it will be automatically traced.
2025/11/12 22:08:58 WARNING mlflow.pyfunc: You have manually traced predict_stream with @mlflow.trace, but this is unnecessary with ResponsesAgent subclasses. You can remove the @mlflow.trace decorator and it will be automatically traced.
Function name main_build__dbdemos_ai_agent__get_customer_billing_and_subscriptions is too long, truncating to 64 characters _build__dbdemos_ai_agent__get_customer_billing_and_subscriptions.


Trace(trace_id=tr-5b0be3dd9ce018caa00b572fc576b83d)

Now log the new agent in the MLflow model registry using `mlflow.pyfunc.log_model()` as in the notebook `02.1_agent evaluation`.

In [0]:
# Agent captures required resources for agent execution, note that it now has the VS index referenced
for r in AGENT.get_resources():
  print(f"Resource: {type(r).__name__}:{r.name}")

Resource: DatabricksServingEndpoint:databricks-claude-3-7-sonnet
Resource: DatabricksFunction:main.dbdemos_ai_agent.calculate_math_expression
Resource: DatabricksFunction:main.dbdemos_ai_agent.get_customer_billing_and_subscriptions
Resource: DatabricksFunction:main.dbdemos_ai_agent.get_customer_by_email


In [0]:
with mlflow.start_run(run_name=model_config.get('config_version_name')):
  logged_agent_info = mlflow.pyfunc.log_model(
    name="agent",
    python_model=agent_eval_path+"/agent.py",
    model_config=conf_path,
    input_example={"input": [{"role": "user", "content": request_example}]},
     # Determine resources (endpoints, fonctions, vs...) to specify for automatic auth passthrough for deployment
    resources=AGENT.get_resources(),
    extra_pip_requirements=["databricks-connect"]
    )

🔗 View Logged Model at: https://xxxx.cloud.databricks.com/ml/experiments/6b46e63d8b9947a09920c46ad4ea44a3/models/m-a3e5e754c8224b96a071267af7a48103?o=1660015457675682
2025/11/12 22:09:08 WARNING mlflow.pyfunc: You have manually traced predict with @mlflow.trace, but this is unnecessary with ResponsesAgent subclasses. You can remove the @mlflow.trace decorator and it will be automatically traced.
2025/11/12 22:09:08 WARNING mlflow.pyfunc: You have manually traced predict_stream with @mlflow.trace, but this is unnecessary with ResponsesAgent subclasses. You can remove the @mlflow.trace decorator and it will be automatically traced.
Function name main_build__dbdemos_ai_agent__get_customer_billing_and_subscriptions is too long, truncating to 64 characters _build__dbdemos_ai_agent__get_customer_billing_and_subscriptions.
2025/11/12 22:09:08 INFO mlflow.pyfunc: Predicting on input example to validate output
2025/11/12 22:09:08 WARNING mlflow.tracing.fluent: No active trace found. Please crea

## 4/ Evaluate our agent against our documents base

Our new model is available! As usual, the next step is to evaluate our dataset to make sure we're improving our answers.


### 4.1/ Generate synthetic eval data

Note that our eval dataset doesn't have any entry on our PDF.

Using Databricks, it's easy to bootstrap our evaluation dataset with synthetic eval data, and then improve this dataset over time.

In [0]:
from databricks.agents.evals import generate_evals_df

docs = spark.table('knowledge_base')
# Describe what our agent is doing
agent_description = """
The Agent is a RAG chatbot that answers technical questions about products such as wifi router, Fiber Installation, network information, but also customer retention strategies or guidelines on social media. The Agent has access to a corpus of Documents, and its task is to answer the user's questions by retrieving the relevant docs from the corpus and synthesizing a helpful, accurate response.
"""

question_guidelines = """
# User personas
- A customer asking question on how to troubleshoot the system, step by step
- An internal agent asking question on internal policies

# Example questions
- How do I troubleshoot Error Code 1001: Invalid Return Authorization when a customer can't submit their return request?
- I'm getting Error Code 1001 when trying to deploy a survey. What could be causing this and how do I fix it?

# Additional Guidelines
- Questions should be succinct, and human-like
"""

# Generate synthetic eval dataset
evals = generate_evals_df(
    docs,
    # The total number of evals to generate. The method attempts to generate evals that have full coverage over the documents
    # provided. If this number is less than the number of documents,
    # some documents will not have any evaluations generated. See "How num_evals is used" below for more details.
    num_evals=10
    ,
    # A set of guidelines that help guide the synthetic generation. These are free-form strings that will be used to prompt the generation.
    agent_description=agent_description,
    question_guidelines=question_guidelines
)
evals["inputs"] = evals["inputs"].apply(lambda x: {"question": x["messages"][0]["content"]})
display(evals)

Generating evaluations:   0%|          | 0/10 evals generated [Elapsed: 00:00, Remaining: ?]

request_id,source_type,source_id,inputs,expectations
8bab4566eaa057669a6958c9ad0bd8f7afcb9ca7e604e71e67581b58d8bbcae3,SYNTHETIC_FROM_DOC,dbfs:/Volumes/main/dbdemos_ai_agent/raw_data/pdf_documentation/sat_survey_2024_manual.pdf,List(What are the steps for the initial configuration of the SAT-SURVEY-2024 system?),"List(List(Login with default admin credentials., Change the default password after the first login., Configure survey parameters (language options, survey templates, respondent limits)., Set up user roles and permissions., Integrate with CRM or customer databases., Test survey deployment with a sample respondent account.), List(List(3.4 Initial Configuration 1. Login with default admin credentials provided in the setup guide. 2. Change default password immediately after first login. 3. Configure survey parameters: language options, survey templates, respondent limits 4. Set up user roles and permissions under Settings > User Management. 5. Integrate with existing CRM or customer databases if applicable 6. Test survey deployment with a sample respondent account., dbfs:/Volumes/main/dbdemos_ai_agent/raw_data/pdf_documentation/sat_survey_2024_manual.pdf)))"
13031d81db91c765fd6825d107093c130079240a2f98b74a397f73753d796ba4,SYNTHETIC_FROM_DOC,dbfs:/Volumes/main/dbdemos_ai_agent/raw_data/pdf_documentation/sat_survey_2024_manual.pdf,List(How do I troubleshoot Error Code 1001: Survey Deployment Failure?),"List(List(Verify server network connectivity., Check CPU and RAM usage on the server., Restart the survey deployment service., Test the survey URL accessibility., Escalate the issue to the network administrator if the problem persists.), List(List(5.1 Error Code 1001: Survey Deployment Failure Cause: Network connectivity issues or server overload. Symptoms: Survey not accessible; deployment status shows 'Failed'. Resolution Steps: 1. Verify server network connectivity: ping the server IP. 2. Check server load via system monitor; ensure CPU and RAM are within normal ranges. 3. Restart the survey deployment service: systemctl restart survey-service (Linux) or restart service via Windows Services. 4. Test survey URL accessibility from client machines. 5. If issue persists, escalate to network administrator., dbfs:/Volumes/main/dbdemos_ai_agent/raw_data/pdf_documentation/sat_survey_2024_manual.pdf)))"
eccbf31adb9ed552974ec14503d04d29fe9c28122cf61f2435236807b514d24c,SYNTHETIC_FROM_DOC,dbfs:/Volumes/main/dbdemos_ai_agent/raw_data/pdf_documentation/return_policy_2024_manual.pdf,List(How do I resolve return shipment problems when the returned equipment is not received or is delayed?),"List(List(Track the shipment using the tracking number., Confirm delivery status with the carrier., Contact the carrier for confirmation if necessary., Verify the accuracy of the address used.), List(List(6.2 Return Shipment Problems Issue: Returned equipment not received or delayed. Diagnosis: Track shipment using provided tracking number; confirm carrier delivery status. Solution: Contact carrier for delivery confirmation; verify address accuracy., dbfs:/Volumes/main/dbdemos_ai_agent/raw_data/pdf_documentation/return_policy_2024_manual.pdf)))"
e49a6558cf36c5c78cf0bfc93b2d8993ad8742a37c924e7ffadefd3e4321bb97,SYNTHETIC_FROM_DOC,dbfs:/Volumes/main/dbdemos_ai_agent/raw_data/pdf_documentation/return_policy_2024_manual.pdf,List(How do I troubleshoot Error Code 1001: Invalid Return Authorization when a customer is unable to proceed with their return?),"List(List(Verify the return authorization number against system records, If missing or invalid, initiate a new return authorization, Ensure the authorization is active and within the valid period, Update the return request with the correct authorization details, Contact technical support if the error persists, Escalate to the Return Policy Supervisor if issues are not resolved within 24 hours), List(List(Error Code 1001: Invalid Return Authorization Description: The return request was submitted without proper a

In [0]:
# Add our synthetic dataset to our MLFLow evaluation dataset
eval_dataset_table_name = f"{catalog}.{dbName}.ai_agent_mlflow_eval"

eval_dataset = mlflow.genai.datasets.get_dataset(eval_dataset_table_name)
eval_dataset.merge_records(evals)
print("Added records to the evaluation dataset.")

Added records to the evaluation dataset.


### 4.2/ Running our evaluation
As previously, let's run our evaluations using the MLFlow dataset. We'll make sure our model still behave properly on the customer-related question, and now perform well on our knowledge-base questions!

In [0]:
from mlflow.genai.scorers import RetrievalGroundedness, RelevanceToQuery, Safety, Guidelines
import pandas as pd

eval_dataset = mlflow.genai.datasets.get_dataset(f"{catalog}.{dbName}.ai_agent_mlflow_eval")

#Get the same scorers as previously (function is defined in _resources/01-setup, similar to the previous step)
scorers = get_scorers()

# Load the model and create a prediction function
loaded_model = mlflow.pyfunc.load_model(f"runs:/{logged_agent_info.run_id}/agent")
def predict_wrapper(question):
    # Format for chat-style models
    model_input = pd.DataFrame({
        "input": [[{"role": "user", "content": question}]]
    })
    response = loaded_model.predict(model_input)
    return response['output'][-1]['content'][-1]['text']
    
print("Running evaluation...")
with mlflow.start_run(run_name='eval_with_retriever'):
    results = mlflow.genai.evaluate(data=eval_dataset, predict_fn=predict_wrapper, scorers=scorers)

2025/11/12 22:10:10 WARNING mlflow.pyfunc: You have manually traced predict with @mlflow.trace, but this is unnecessary with ResponsesAgent subclasses. You can remove the @mlflow.trace decorator and it will be automatically traced.
2025/11/12 22:10:10 WARNING mlflow.pyfunc: You have manually traced predict_stream with @mlflow.trace, but this is unnecessary with ResponsesAgent subclasses. You can remove the @mlflow.trace decorator and it will be automatically traced.
Function name main_build__dbdemos_ai_agent__get_customer_billing_and_subscriptions is too long, truncating to 64 characters _build__dbdemos_ai_agent__get_customer_billing_and_subscriptions.


Running evaluation...


2025/11/12 22:10:11 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2025/11/12 22:10:11 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset.
2025/11/12 22:10:11 WARNING mlflow.tracing.fluent: No active trace found. Please create a span using `mlflow.start_span` or `@mlflow.trace` before calling `mlflow.update_current_trace`.
2025/11/12 22:10:17 WARNING mlflow.tracing.fluent: No active trace found. Please create a span using `mlflow.start_span` or `@mlflow.trace` before calling `mlflow.update_current_trace`.


Evaluating:   0%|          | 0/123 [Elapsed: 00:00, Remaining: ?] 

<!DOCTYPE html>
 
 
 Evaluation output 
 
 
 
 
 
 
 
 
 View evaluation results.

[Trace(trace_id=tr-68155e7c0eb3557594f6ed1ecce929eb), Trace(trace_id=tr-2351fe504a45bba1d5ed124f7d455670), Trace(trace_id=tr-41c6f8d91d8d05fca3adc099b66bb68b), Trace(trace_id=tr-093f5acf8da68a44bc9d282245f5c2fb), Trace(trace_id=tr-0aaebb97d8951da12c210dd122324af1), Trace(trace_id=tr-8a994a91deac27465d4ecccb63f08be4), Trace(trace_id=tr-0f988d4160d03b12bb48d5ab31300706), Trace(trace_id=tr-3ec1db991bb7bd4806eb5a4a7232c1ab), Trace(trace_id=tr-40d8090a7ffcbf1cc1c5bbbd010492e2), Trace(trace_id=tr-70d699aa72b67029ff198bc01db27032)]

### 4.3/ Deploy the final model! 

We're good to go. Let's deploy our model to UC and update our endpoint with the latest version!

In [0]:
from mlflow import MlflowClient
MODEL_NAME = "dbdemos_ai_agent_demo"
UC_MODEL_NAME = f"{catalog}.{dbName}.{MODEL_NAME}"

# register the model to UC
client = MlflowClient()
uc_registered_model_info = mlflow.register_model(model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME, tags={"model": "customer_support_agent", "model_version": "with_retriever"})

client.set_registered_model_alias(name=UC_MODEL_NAME, alias="model-to-deploy", version=uc_registered_model_info.version)
displayHTML(f'<a href="/explore/data/models/{catalog}/{dbName}/{MODEL_NAME}" target="_blank">Open Unity Catalog to see Registered Agent</a>')

Registered model 'main.dbdemos_ai_agent.dbdemos_ai_agent_demo' already exists. Creating a new version of this model...
🔗 Created version '23' of model 'main.dbdemos_ai_agent.dbdemos_ai_agent_demo': https://xxxx.cloud.databricks.com/explore/data/models/main_build/dbdemos_ai_agent/dbdemos_ai_agent_demo/version/23?o=1660015457675682


Open Unity Catalog to see Registered Agent

In [0]:
from databricks import agents
# Deploy the model to the review app and a model serving endpoint
endpoint_name = f'{MODEL_NAME}_{catalog}_{db}'[:60]

if len(agents.get_deployments(model_name=UC_MODEL_NAME, model_version=uc_registered_model_info.version)) == 0:
  agents.deploy(UC_MODEL_NAME, uc_registered_model_info.version, endpoint_name=endpoint_name, tags = {"project": "dbdemos"})


    Deployment of main.dbdemos_ai_agent.dbdemos_ai_agent_demo version 23 initiated.  This can take up to 15 minutes and the Review App & Query Endpoint will not work until this deployment finishes.

    View status: https://xxxx.cloud.databricks.com/ml/endpoints/dbdemos_ai_agent_demo_main_build_dbdemos_ai_agent
    Review App: https://xxxx.cloud.databricks.com/ml/review-v2/chat?endpoint=dbdemos_ai_agent_demo_main_build_dbdemos_ai_agent

You can refer back to the links above from the endpoint detail page at https://xxxx.cloud.databricks.com/ml/endpoints/dbdemos_ai_agent_demo_main_build_dbdemos_ai_agent.


## Next: deploy our chatbout within a Databricks Application

Now that our agent is ready, let's deploy a GradIO application to serve its content to our end users. 

Open [04-deploy-app/04-Deploy-Frontend-Lakehouse-App]($../04-deploy-app/04-Deploy-Frontend-Lakehouse-App) !